### From Silver To Gold: Aggregation and KPI Tables 

In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable 

#### Widgets

In [0]:
dbutils.widgets.text("catalog_name","ecommerce","Catalog Name")
dbutils.widgets.text("storage_account_name","rwecommerceadlsdevus001","Storage Account Name")
dbutils.widgets.text("container_name","ecom-raw-data")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

print(catalog_name, storage_account_name, container_name)

ecommerce rwecommerceadlsdevus001 ecom-raw-data


In [0]:
# readChangeFeed flag is used to read the change feed (_change_type column mainly)
df = spark.readStream \
.format("delta") \
.option("readChangeFeed", "true") \
.table(f"{catalog_name}.silver.slv_order_items")

In [0]:
df_union = df.filter("_change_type IN ('insert', 'update_postimage')")

In [0]:
df_union = df_union.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
    )

# 2) Add discount_amount (discount_pct is already numeric, e.g., 21 -> 21%)
df_union = df_union.withColumn(
    "discount_amount",
    F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
)

# 3) Add sale_amount = gross - discount
df_union = df_union.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
)

# add date id
df_union = df_union.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType()))  # Create date_key

# Coupon flag
#  coupon flag = 1 if coupon_code is not null else 0
df_union = df_union.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
     .otherwise(F.lit(0))
)  

In [0]:
orders_gold_df = df_union.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount")
)

### Write to Gold Table

In [0]:
gold_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/gold/fact_order_items/"
dbutils.fs.mkdirs(gold_checkpoint_path)
print(gold_checkpoint_path)

gold_table_name = f"{catalog_name}.gold.gld_fact_order_items"
silver_table_name = f"{catalog_name}.silver.slv_order_items"

def build_gold_df(input_df):
    return input_df.filter(
        F.col("_change_type").isin("insert", "update_postimage")
    ) if "_change_type" in input_df.columns else input_df


def transform_gold_df(input_df):
    transformed_df = build_gold_df(input_df).withColumn(
        "gross_amount",
        F.col("quantity") * F.col("unit_price")
    ).withColumn(
        "discount_amount",
        F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
    ).withColumn(
        "sale_amount",
        F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
    ).withColumn(
        "date_id",
        F.date_format(F.col("dt"), "yyyyMMdd").cast(IntegerType())
    ).withColumn(
        "coupon_flag",
        F.when(F.col("coupon_code").isNotNull(), F.lit(1)).otherwise(F.lit(0))
    )

    return transformed_df.select(
        F.col("date_id"),
        F.col("dt").alias("transaction_date"),
        F.col("order_ts").alias("transaction_ts"),
        F.col("order_id").alias("transaction_id"),
        F.col("customer_id"),
        F.col("item_seq").alias("seq_no"),
        F.col("product_id"),
        F.col("channel"),
        F.col("coupon_code"),
        F.col("coupon_flag"),
        F.col("unit_price_currency"),
        F.col("quantity"),
        F.col("unit_price"),
        F.col("gross_amount"),
        F.col("discount_pct").alias("discount_percent"),
        F.col("discount_amount"),
        F.col("tax_amount"),
        F.col("sale_amount").alias("net_amount")
    )

def upsert_to_gold(microBatchDF, batchId):
    deltaTable = DeltaTable.forName(spark, gold_table_name)
    deltaTable.alias("gold_table").merge(
        microBatchDF.alias("batch_table"),
        "gold_table.transaction_id = batch_table.transaction_id AND gold_table.seq_no = batch_table.seq_no",
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

if not spark.catalog.tableExists(gold_table_name):
    print("creating new table")
    transform_gold_df(spark.table(silver_table_name)).write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
    spark.sql(
        f"ALTER TABLE {gold_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
    )
else:
    cdc_orders_gold_df = transform_gold_df(
        spark.readStream.format("delta").option("readChangeFeed", "true").option(
            "startingVersion", 1
        ).table(silver_table_name)
    )

    cdc_orders_gold_df.writeStream.foreachBatch(upsert_to_gold).format("delta").option(
        "checkpointLocation", gold_checkpoint_path
    ).option("mergeSchema", "true").outputMode("update").trigger(availableNow=True).start().awaitTermination()

abfss://ecom-raw-data@rwecommerceadlsdevus001.dfs.core.windows.net/checkpoint/gold/fact_order_items/


In [0]:
spark.sql(f"SELECT count(*) FROM {catalog_name}.gold.gld_fact_order_items").show()

+--------+
|count(*)|
+--------+
| 1193360|
+--------+



In [0]:
spark.sql(f"SELECT max(transaction_date) FROM {catalog_name}.gold.gld_fact_order_items").show()

+---------------------+
|max(transaction_date)|
+---------------------+
|           2025-09-03|
+---------------------+



In [0]:
spark.sql(f"SELECT * FROM {catalog_name}.gold.gld_fact_order_items LIMIT 5").show()

+--------+----------------+-------------------+--------------+----------------+------+-------------+-------+-----------+-----------+-------------------+--------+----------+------------+----------------+---------------+----------+----------+
| date_id|transaction_date|     transaction_ts|transaction_id|     customer_id|seq_no|   product_id|channel|coupon_code|coupon_flag|unit_price_currency|quantity|unit_price|gross_amount|discount_percent|discount_amount|tax_amount|net_amount|
+--------+----------------+-------------------+--------------+----------------+------+-------------+-------+-----------+-----------+-------------------+--------+----------+------------+----------------+---------------+----------+----------+
|20250830|      2025-08-30|2025-08-30 07:45:21|        676515|CUST000000038846|     1|2000000361970|Website|     fest20|          1|                CAD|       1|      80.0|        80.0|             8.0|              7|         4|      77.0|
|20250830|      2025-08-30|2025-08-3